# Exposing Tools and Resources via MCP

The **Model Context Protocol (MCP)** gives agents a standard way to discover and use capabilities hosted outside the application process. An MCP **server** advertises what it offers; an MCP **client** (inside a host app) discovers and invokes those capabilities on behalf of an agent.

Two capability types matter most when you build servers:

| Capability | Purpose | Agent usage |
|------------|---------|-------------|
| **Tool** | Perform an action (lookup, create, calculate) | Model chooses tool + arguments; server executes |
| **Resource** | Expose read-only context (policies, files, snapshots) | Host or model reads URI content into context |

This notebook implements a **self-contained MCP server and client** in plain Python — no external MCP SDK required. The scenario is **ShopCo Support Hub**: an agent needs order lookups (tools) and policy documents (resources) to answer customer tickets.

You will learn how to:

1. Design tool schemas models can reliably call
2. Expose static and templated resources with stable URIs
3. Implement `tools/list`, `tools/call`, `resources/list`, and `resources/read`
4. Wire a client that discovers capabilities and runs an offline agent loop
5. Apply security annotations and validation before execution

Everything runs offline. An optional cell at the end shows how the same discovered tools would feed a live LLM.

In [ ]:
"""
ShopCo Support Hub — MCP tools + resources lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- ShopCo backing data (simulated CRM / policy store) ---

ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {
        "order_id": "ORD-1001",
        "customer": "alice@example.com",
        "status": "shipped",
        "total_usd": 89.50,
        "items": ["Wireless Mouse", "USB-C Hub"],
        "carrier": "UPS",
        "tracking": "1Z999AA10123456784",
    },
    "ORD-1002": {
        "order_id": "ORD-1002",
        "customer": "bob@example.com",
        "status": "processing",
        "total_usd": 42.00,
        "items": ["Desk Lamp"],
        "carrier": None,
        "tracking": None,
    },
}

POLICIES: dict[str, str] = {
    "returns": """# ShopCo Returns Policy

- 30-day return window from delivery date.
- Items must be unused and in original packaging.
- Refunds processed within 5–7 business days after warehouse receipt.
- Final-sale items are non-returnable.
""",
    "shipping": """# ShopCo Shipping Policy

- Free standard shipping on orders over $50.
- Express shipping available at checkout.
- Tracking numbers emailed when the carrier scans the package.
""",
    "billing": """# ShopCo Billing Policy

- Charges appear as SHOPCO* on card statements.
- Partial refunds may take up to two billing cycles to display.
- Contact support with order ID for charge disputes.
""",
}

REFUND_LOG: list[dict[str, Any]] = []

print(f"ShopCo corpus: {len(ORDERS)} orders, {len(POLICIES)} policy documents")

---

## 1. Tools vs Resources vs Prompts

MCP servers can expose three capability families. This notebook focuses on **tools** and **resources** — the pair you use most when connecting agents to business systems.

| Type | Mutates state? | Typical examples | Discovery method |
|------|----------------|------------------|----------------|
| **Tool** | Often yes | `get_order`, `create_refund_request` | `tools/list` |
| **Resource** | No (read-only) | Policy markdown, order snapshot URI | `resources/list`, `resources/templates/list` |
| **Prompt** | No | Reusable prompt templates | `prompts/list` |

**Design rule:** Put **facts the model should cite** in resources. Put **operations that change or fetch live data** in tools. Mixing both in one oversized tool forces the model to guess whether it is reading or acting.

```
  Agent (in host)
       │
       ▼
  MCP Client ──JSON-RPC──▶ MCP Server
       │                        │
       │                   tools/list  → actions
       │                   resources/list → context URIs
       ▼
  LLM sees tool schemas + optional resource text
```

---

## 2. MCP Message Shape (JSON-RPC)

MCP rides on **JSON-RPC 2.0**. Requests have `method`, optional `params`, and an `id`. Responses return `result` or `error`.

Core methods for tools and resources:

| Method | Direction | Purpose |
|--------|-----------|--------|
| `initialize` | client → server | Handshake, protocol version |
| `tools/list` | client → server | Discover tool manifests |
| `tools/call` | client → server | Execute a tool by name |
| `resources/list` | client → server | List static resource URIs |
| `resources/templates/list` | client → server | List URI templates |
| `resources/read` | client → server | Fetch resource body by URI |

We implement a minimal in-process transport so you can see exact payloads without running a separate process.

In [ ]:
@dataclass
class JsonRpcRequest:
    method: str
    params: dict[str, Any] = field(default_factory=dict)
    id: str = field(default_factory=lambda: str(uuid.uuid4()))

    def to_dict(self) -> dict[str, Any]:
        return {"jsonrpc": "2.0", "method": self.method, "params": self.params, "id": self.id}


@dataclass
class JsonRpcResponse:
    id: str
    result: Any = None
    error: dict[str, Any] | None = None

    def to_dict(self) -> dict[str, Any]:
        payload: dict[str, Any] = {"jsonrpc": "2.0", "id": self.id}
        if self.error is not None:
            payload["error"] = self.error
        else:
            payload["result"] = self.result
        return payload


PROTOCOL_VERSION = "2024-11-05"
SERVER_INFO = {"name": "shopco-support-mcp", "version": "1.0.0"}

print("JSON-RPC envelope ready")

---

## 3. Designing MCP Tool Schemas

Each tool advertises a JSON Schema `inputSchema`. Good schemas are **narrow**, **typed**, and **honest** in the description.

Checklist for ShopCo tools:

1. **One verb per tool** — `get_order`, not `handle_order_stuff`.
2. **Required fields explicit** — `order_id` in `required`.
3. **Enums for closed sets** — refund reasons, carriers.
4. **Description says when NOT to call** — reduces wrong-tool selection.
5. **Destructive tools annotated** — `destructive: true` in metadata hints hosts to add approval gates.

In [ ]:
SHOPCO_TOOLS: list[dict[str, Any]] = [
    {
        "name": "get_order",
        "description": (
            "Fetch live order details by order ID. "
            "Do NOT use for policy questions — read policy resources instead."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "ShopCo order ID, e.g. ORD-1001",
                    "pattern": "^ORD-[0-9]{4}$",
                }
            },
            "required": ["order_id"],
            "additionalProperties": False,
        },
        "annotations": {"readOnlyHint": True},
    },
    {
        "name": "create_refund_request",
        "description": (
            "Open a refund request ticket. Does not instantly charge the card — "
            "creates a workflow item for human review when amount exceeds policy."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "reason": {
                    "type": "string",
                    "enum": ["damaged", "wrong_item", "late_delivery", "changed_mind"],
                },
                "amount_usd": {"type": "number", "minimum": 0.01},
            },
            "required": ["order_id", "reason", "amount_usd"],
            "additionalProperties": False,
        },
        "annotations": {"destructiveHint": True},
    },
]

print("Tool manifests:")
for t in SHOPCO_TOOLS:
    print(f"  - {t['name']}: {len(t['inputSchema'].get('required', []))} required fields")

---

## 4. Designing MCP Resources

**Resources** are addressable content identified by URIs. They are ideal for policy text, configuration snapshots, and documents that change infrequently.

Two patterns:

| Pattern | Example URI | When to use |
|---------|---------------|-------------|
| **Static list** | `shopco://policies/returns` | Fixed catalog entries |
| **URI template** | `shopco://orders/{order_id}` | Parameterized read-only snapshots |

Resources should declare `mimeType` (e.g. `text/markdown`) so hosts know how to render or chunk them.

In [ ]:
SHOPCO_RESOURCES: list[dict[str, Any]] = [
    {
        "uri": f"shopco://policies/{key}",
        "name": f"ShopCo {key.title()} Policy",
        "description": f"Official ShopCo {key} policy (read-only)",
        "mimeType": "text/markdown",
    }
    for key in POLICIES
]

SHOPCO_RESOURCE_TEMPLATES: list[dict[str, Any]] = [
    {
        "uriTemplate": "shopco://orders/{order_id}",
        "name": "Order snapshot",
        "description": "Read-only JSON snapshot of an order (prefer get_order tool for live status)",
        "mimeType": "application/json",
    }
]

print("Static resources:", [r["uri"] for r in SHOPCO_RESOURCES])
print("Templates:", [t["uriTemplate"] for t in SHOPCO_RESOURCE_TEMPLATES])

---

## 5. Input Validation Layer

Never trust model-generated arguments. Validate against the tool's `inputSchema` **before** touching business logic — same discipline as REST API request validation.

In [ ]:
class SchemaValidationError(Exception):
    pass


def validate_tool_input(schema: dict[str, Any], arguments: dict[str, Any]) -> None:
    if schema.get("type") != "object":
        raise SchemaValidationError("root must be object")
    if schema.get("additionalProperties") is False:
        extra = set(arguments) - set(schema.get("properties", {}))
        if extra:
            raise SchemaValidationError(f"unexpected fields: {sorted(extra)}")
    for req in schema.get("required", []):
        if req not in arguments:
            raise SchemaValidationError(f"missing required field: {req}")
    props = schema.get("properties", {})
    for key, spec in props.items():
        if key not in arguments:
            continue
        val = arguments[key]
        expected = spec.get("type")
        if expected == "string" and not isinstance(val, str):
            raise SchemaValidationError(f"{key} must be string")
        if expected == "number" and not isinstance(val, (int, float)):
            raise SchemaValidationError(f"{key} must be number")
        if "enum" in spec and val not in spec["enum"]:
            raise SchemaValidationError(f"{key} not in enum {spec['enum']}")
        if "pattern" in spec and isinstance(val, str):
            if not re.match(spec["pattern"], val):
                raise SchemaValidationError(f"{key} does not match pattern {spec['pattern']}")
        if "minimum" in spec and isinstance(val, (int, float)) and val < spec["minimum"]:
            raise SchemaValidationError(f"{key} below minimum {spec['minimum']}")


# quick sanity checks
validate_tool_input(SHOPCO_TOOLS[0]["inputSchema"], {"order_id": "ORD-1001"})
try:
    validate_tool_input(SHOPCO_TOOLS[0]["inputSchema"], {"order_id": "BAD"})
except SchemaValidationError as e:
    print("Caught expected validation error:", e)
print("Schema validator OK")

---

## 6. Tool Handler Implementations

Handlers are plain Python functions registered by tool name. They return **structured content** the client wraps in MCP `content` blocks.

In [ ]:
REFUND_AUTO_APPROVE_THRESHOLD_USD = 25.00


def handle_get_order(arguments: dict[str, Any]) -> dict[str, Any]:
    order_id = arguments["order_id"]
    order = ORDERS.get(order_id)
    if not order:
        return {"isError": True, "content": [{"type": "text", "text": f"Order {order_id} not found"}]}
    return {"content": [{"type": "text", "text": pretty(order)}]}


def handle_create_refund_request(arguments: dict[str, Any]) -> dict[str, Any]:
    order_id = arguments["order_id"]
    if order_id not in ORDERS:
        return {"isError": True, "content": [{"type": "text", "text": f"Order {order_id} not found"}]}
    ticket_id = f"RF-{uuid.uuid4().hex[:8].upper()}"
    needs_review = arguments["amount_usd"] > REFUND_AUTO_APPROVE_THRESHOLD_USD
    record = {
        "ticket_id": ticket_id,
        "order_id": order_id,
        "reason": arguments["reason"],
        "amount_usd": arguments["amount_usd"],
        "status": "pending_human_review" if needs_review else "auto_approved",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    REFUND_LOG.append(record)
    return {"content": [{"type": "text", "text": pretty(record)}]}


TOOL_HANDLERS: dict[str, Callable[[dict[str, Any]], dict[str, Any]]] = {
    "get_order": handle_get_order,
    "create_refund_request": handle_create_refund_request,
}

print("Registered handlers:", list(TOOL_HANDLERS.keys()))

---

## 7. Resource Reader Implementations

Resource readers map a URI to bytes/text. Template URIs are resolved by substituting path parameters, then reading the backing store.

In [ ]:
def read_policy_resource(uri: str) -> dict[str, Any]:
    # shopco://policies/{name}
    prefix = "shopco://policies/"
    if not uri.startswith(prefix):
        raise ValueError(f"unsupported policy URI: {uri}")
    key = uri[len(prefix):]
    if key not in POLICIES:
        raise ValueError(f"unknown policy: {key}")
    return {
        "uri": uri,
        "mimeType": "text/markdown",
        "text": POLICIES[key],
    }


def read_order_resource(uri: str) -> dict[str, Any]:
    # shopco://orders/{order_id}
    m = re.match(r"^shopco://orders/(ORD-[0-9]{4})$", uri)
    if not m:
        raise ValueError(f"unsupported order URI: {uri}")
    order_id = m.group(1)
    order = ORDERS.get(order_id)
    if not order:
        raise ValueError(f"order not found: {order_id}")
    return {
        "uri": uri,
        "mimeType": "application/json",
        "text": pretty(order),
    }


def read_resource(uri: str) -> dict[str, Any]:
    if uri.startswith("shopco://policies/"):
        return read_policy_resource(uri)
    if uri.startswith("shopco://orders/"):
        return read_order_resource(uri)
    raise ValueError(f"unknown resource URI: {uri}")


sample = read_resource("shopco://policies/returns")
print(sample["text"][:80], "...")

---

## 8. Building the MCP Server

`ShopCoMCPServer` dispatches JSON-RPC methods. In production this class runs behind stdio, SSE, or WebSocket transport; here we call `handle()` directly.

In [ ]:
class ShopCoMCPServer:
    """Minimal MCP server exposing ShopCo tools and resources."""

    def __init__(self) -> None:
        self._initialized = False
        self._tool_index = {t["name"]: t for t in SHOPCO_TOOLS}

    def handle(self, request: JsonRpcRequest) -> JsonRpcResponse:
        try:
            if request.method == "initialize":
                return self._initialize(request)
            if not self._initialized:
                return JsonRpcResponse(id=request.id, error={"code": -32002, "message": "not initialized"})
            dispatch = {
                "tools/list": self._tools_list,
                "tools/call": self._tools_call,
                "resources/list": self._resources_list,
                "resources/templates/list": self._resource_templates_list,
                "resources/read": self._resources_read,
            }
            handler = dispatch.get(request.method)
            if not handler:
                return JsonRpcResponse(id=request.id, error={"code": -32601, "message": f"unknown method {request.method}"})
            return handler(request)
        except SchemaValidationError as exc:
            return JsonRpcResponse(id=request.id, error={"code": -32602, "message": str(exc)})
        except Exception as exc:
            return JsonRpcResponse(id=request.id, error={"code": -32000, "message": str(exc)})

    def _initialize(self, request: JsonRpcRequest) -> JsonRpcResponse:
        self._initialized = True
        return JsonRpcResponse(
            id=request.id,
            result={
                "protocolVersion": PROTOCOL_VERSION,
                "capabilities": {"tools": {}, "resources": {}},
                "serverInfo": SERVER_INFO,
            },
        )

    def _tools_list(self, request: JsonRpcRequest) -> JsonRpcResponse:
        return JsonRpcResponse(id=request.id, result={"tools": SHOPCO_TOOLS})

    def _tools_call(self, request: JsonRpcRequest) -> JsonRpcResponse:
        name = request.params.get("name")
        arguments = request.params.get("arguments", {})
        if name not in self._tool_index:
            return JsonRpcResponse(id=request.id, error={"code": -32602, "message": f"unknown tool: {name}"})
        schema = self._tool_index[name]["inputSchema"]
        validate_tool_input(schema, arguments)
        handler = TOOL_HANDLERS[name]
        tool_result = handler(arguments)
        return JsonRpcResponse(id=request.id, result=tool_result)

    def _resources_list(self, request: JsonRpcRequest) -> JsonRpcResponse:
        return JsonRpcResponse(id=request.id, result={"resources": SHOPCO_RESOURCES})

    def _resource_templates_list(self, request: JsonRpcRequest) -> JsonRpcResponse:
        return JsonRpcResponse(id=request.id, result={"resourceTemplates": SHOPCO_RESOURCE_TEMPLATES})

    def _resources_read(self, request: JsonRpcRequest) -> JsonRpcResponse:
        uri = request.params.get("uri")
        if not uri:
            return JsonRpcResponse(id=request.id, error={"code": -32602, "message": "uri required"})
        body = read_resource(uri)
        return JsonRpcResponse(id=request.id, result={"contents": [body]})


server = ShopCoMCPServer()
init_resp = server.handle(JsonRpcRequest(method="initialize", params={"protocolVersion": PROTOCOL_VERSION}))
print("Initialize:", pretty(init_resp.to_dict()))

---

## 9. Building the MCP Client

The client performs handshake, caches discovered tools/resources, and exposes ergonomic methods for the agent loop.

In [ ]:
class MCPClient:
    def __init__(self, server: ShopCoMCPServer) -> None:
        self.server = server
        self.tools: list[dict[str, Any]] = []
        self.resources: list[dict[str, Any]] = []
        self.resource_templates: list[dict[str, Any]] = []
        self._call_log: list[dict[str, Any]] = []

    def connect(self) -> dict[str, Any]:
        resp = self._request("initialize", {"protocolVersion": PROTOCOL_VERSION})
        self.tools = self._request("tools/list")["tools"]
        self.resources = self._request("resources/list")["resources"]
        self.resource_templates = self._request("resources/templates/list")["resourceTemplates"]
        return resp

    def _request(self, method: str, params: dict[str, Any] | None = None) -> Any:
        req = JsonRpcRequest(method=method, params=params or {})
        resp = self.server.handle(req)
        payload = resp.to_dict()
        self._call_log.append({"method": method, "params": params, "response": payload})
        if "error" in payload:
            raise RuntimeError(f"MCP error on {method}: {payload['error']}")
        return payload["result"]

    def call_tool(self, name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        return self._request("tools/call", {"name": name, "arguments": arguments})

    def read_resource(self, uri: str) -> str:
        result = self._request("resources/read", {"uri": uri})
        return result["contents"][0]["text"]

    def tools_for_llm(self) -> list[dict[str, Any]]:
        """Shape manifests for OpenAI-style function / tool calling APIs."""
        return [
            {
                "type": "function",
                "function": {
                    "name": t["name"],
                    "description": t["description"],
                    "parameters": t["inputSchema"],
                },
            }
            for t in self.tools
        ]


client = MCPClient(server)
client.connect()
print(f"Discovered {len(client.tools)} tools, {len(client.resources)} resources")
print("Tool names:", [t["name"] for t in client.tools])

---

## 10. Prefetching Resources into Agent Context

Hosts often **prefetch** relevant resources before the model turn — especially policies the agent must cite accurately. Tools then fetch live operational data on demand.

Pattern for ticket triage:

1. Read `shopco://policies/returns` into system context.
2. Call `get_order` when the customer mentions an order ID.
3. Only call `create_refund_request` after policy check passes.

In [ ]:
def build_support_context(client: MCPClient, policy_keys: list[str]) -> str:
    blocks = []
    for key in policy_keys:
        uri = f"shopco://policies/{key}"
        blocks.append(f"### Resource: {uri}\n{client.read_resource(uri)}")
    return "\n\n".join(blocks)


context = build_support_context(client, ["returns", "billing"])
print(context[:300], "\n... [truncated]")

---

## 11. Offline Agent Loop Using Discovered Tools

`OfflineSupportAgent` simulates an LLM planner: given a ticket, it decides which MCP tools to call based on simple rules — the same dispatch pattern a real model would drive.

In [ ]:
@dataclass
class Ticket:
    ticket_id: str
    customer_email: str
    body: str


@dataclass
class AgentTrace:
    steps: list[dict[str, Any]] = field(default_factory=list)

    def log(self, kind: str, detail: str, **extra: Any) -> None:
        self.steps.append({"kind": kind, "detail": detail, **extra})


class OfflineSupportAgent:
    def __init__(self, client: MCPClient) -> None:
        self.client = client

    def handle_ticket(self, ticket: Ticket) -> dict[str, Any]:
        trace = AgentTrace()
        policies = build_support_context(self.client, ["returns"])
        trace.log("resource_prefetch", "loaded returns policy", chars=len(policies))

        order_id = self._extract_order_id(ticket.body)
        if not order_id:
            return {"status": "needs_info", "message": "Please provide your order ID (ORD-XXXX).", "trace": trace.steps}

        order_result = self.client.call_tool("get_order", {"order_id": order_id})
        trace.log("tool_call", "get_order", result=order_result)
        if order_result.get("isError"):
            return {"status": "error", "message": order_result["content"][0]["text"], "trace": trace.steps}

        if "refund" in ticket.body.lower():
            refund_result = self.client.call_tool(
                "create_refund_request",
                {"order_id": order_id, "reason": "changed_mind", "amount_usd": 89.50},
            )
            trace.log("tool_call", "create_refund_request", result=refund_result)
            text = refund_result["content"][0]["text"]
            return {"status": "refund_opened", "message": text, "trace": trace.steps}

        if "where is" in ticket.body.lower() or "tracking" in ticket.body.lower():
            order_json = json.loads(order_result["content"][0]["text"])
            tracking = order_json.get("tracking") or "not yet assigned"
            return {
                "status": "answered",
                "message": f"Order {order_id} status: {order_json['status']}. Tracking: {tracking}.",
                "trace": trace.steps,
            }

        return {"status": "answered", "message": "How can I help with your order?", "trace": trace.steps}

    @staticmethod
    def _extract_order_id(text: str) -> str | None:
        m = re.search(r"ORD-[0-9]{4}", text.upper())
        return m.group(0) if m else None


agent = OfflineSupportAgent(client)

t1 = Ticket("T-001", "alice@example.com", "Where is my order ORD-1001? Tracking please.")
t2 = Ticket("T-002", "alice@example.com", "I want a refund for ORD-1001, changed my mind.")

for t in [t1, t2]:
    outcome = agent.handle_ticket(t)
    print(f"\n=== {t.ticket_id} → {outcome['status']} ===")
    print(outcome["message"])
    print("Trace steps:", [s["kind"] for s in outcome["trace"]])

---

## 12. Tools vs Resources — Decision Matrix

| Customer question | Use resource | Use tool |
|-------------------|--------------|----------|
| "What is your return window?" | `shopco://policies/returns` | — |
| "Status of ORD-1001?" | — | `get_order` |
| "Show me a snapshot for audit" | `shopco://orders/ORD-1001` | optional |
| "Start my refund" | read policy first | `create_refund_request` |

Duplicating order JSON as both a resource template **and** `get_order` is intentional pedagogy: resources suit **stable context injection**; tools suit **authoritative live queries** and side effects.

---

## 13. Security and Host Responsibilities

MCP servers expose power; **hosts** must enforce policy:

| Concern | Server responsibility | Host responsibility |
|---------|----------------------|---------------------|
| Schema honesty | Accurate `inputSchema` | Validate before call (defense in depth) |
| Destructive ops | `destructiveHint` annotation | Human approval gate |
| Data scope | Return only authorized rows | Auth token → server identity |
| Prompt injection via resources | Sanitize file sources | Treat resource text as untrusted input |
| Audit | Log tool name + args (redacted) | Central observability |

Never expose raw SQL or shell as MCP tools without strict sandboxing.

In [ ]:
def host_policy_check(tool_manifest: dict[str, Any], arguments: dict[str, Any]) -> str | None:
    """Return error message if host blocks the call, else None."""
    annotations = tool_manifest.get("annotations", {})
    if annotations.get("destructiveHint") and arguments.get("amount_usd", 0) > 100:
        return "Host blocked: refund over $100 requires human approval"
    return None


refund_tool = next(t for t in client.tools if t["name"] == "create_refund_request")
blocked = host_policy_check(refund_tool, {"order_id": "ORD-1001", "reason": "damaged", "amount_usd": 150})
print("Host gate:", blocked or "allowed")

---

## 14. Observability — MCP Call Log

Production hosts log every JSON-RPC round trip: latency, method, redacted params, and error codes. The client below already captures a call log you can export.

In [ ]:
print(f"MCP call log entries: {len(client._call_log)}")
for entry in client._call_log[-4:]:
    method = entry["method"]
    err = entry["response"].get("error")
    status = "ERROR" if err else "OK"
    print(f"  [{status}] {method}")

---

## 15. Hands-On — Resource Read + Tool Call Chain

Walk through the exact JSON payloads for one support scenario end to end.

In [ ]:
demo_client = MCPClient(ShopCoMCPServer())
demo_client.connect()

# 1) resources/read
policy_text = demo_client.read_resource("shopco://policies/shipping")
print("Policy excerpt:", policy_text.splitlines()[0])

# 2) resources/read via template URI
snapshot = demo_client.read_resource("shopco://orders/ORD-1002")
print("Order snapshot:", snapshot)

# 3) tools/call
live = demo_client.call_tool("get_order", {"order_id": "ORD-1002"})
print("Live tool result:", live["content"][0]["text"])

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| One mega-tool does read + write + policy | Model calls wrong mode | Split tools vs resources |
| Vague tool descriptions | Frequent wrong-tool errors | Say when NOT to use |
| No `additionalProperties: false` | Mystery keys slip through | Tight JSON Schema |
| Resources for live mutable state | Stale answers | Prefer tools for live queries |
| Skipping host validation | Server compromise = full access | AuthZ at host + server |
| Returning unstructured prose only | Downstream parsing breaks | Structured `content` blocks |

---

## 17. Server Capability Checklist

Before shipping an MCP server to production:

- [ ] Every tool has `inputSchema` with `required` and typed properties
- [ ] Destructive tools carry `destructiveHint`
- [ ] Read-only lookups carry `readOnlyHint` where appropriate
- [ ] Resources use stable URI scheme (`shopco://...`)
- [ ] `resources/read` returns correct `mimeType`
- [ ] Errors use `isError` on tool results or JSON-RPC `error` codes
- [ ] Server logs redacted arguments
- [ ] Version pinned in `serverInfo`

---

## 18. Optional Live LLM with Discovered Tools

Set `USE_LIVE_LLM = True` to pass `client.tools_for_llm()` to the OpenAI API. The MCP discovery step stays identical — only the planner changes from rules to a model.

In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    from openai import OpenAI

    llm_client = OpenAI()
    live_mcp = MCPClient(ShopCoMCPServer())
    live_mcp.connect()
    tools_payload = live_mcp.tools_for_llm()
    policy_ctx = live_mcp.read_resource("shopco://policies/returns")

    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"You are ShopCo support. Policy:\n{policy_ctx}"},
            {"role": "user", "content": "Can I return order ORD-1001? I changed my mind."},
        ],
        tools=tools_payload,
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = live_mcp.call_tool(tc.function.name, args)
            print(tc.function.name, "→", result)
    else:
        print(msg.content)
else:
    print("Offline mode — set USE_LIVE_LLM = True to exercise live tool discovery.")
    print("Discovered tool schemas ready:", [t["function"]["name"] for t in client.tools_for_llm()])

---

## 19. Quiz

1. When should you expose data as an MCP **resource** instead of a **tool**?
2. What JSON-RPC methods discover tools vs read resource content?
3. Why validate `arguments` on the server even if the host already validated?
4. What is the purpose of `uriTemplate` in `resources/templates/list`?
5. How does `tools_for_llm()` bridge MCP manifests to OpenAI-style tool calling?

---

## 20. Summary

**MCP tools** expose actions with JSON Schema contracts; **MCP resources** expose read-only context at stable URIs. A server implements `tools/list`, `tools/call`, `resources/list`, and `resources/read`; a client discovers capabilities and feeds them to an agent loop.

For **ShopCo Support Hub**:

- Policies → resources (`shopco://policies/...`)
- Order lookup and refunds → tools (`get_order`, `create_refund_request`)
- Host prefetches policies, validates args, gates destructive calls

The plain-Python `ShopCoMCPServer` and `MCPClient` mirror production wire behavior. Swap the in-process `handle()` call for stdio/SSE transport and you have a deployable MCP integration path.